In [5]:
import pandas as pd

b = pd.read_parquet("../data/interim/balls.parquet")

print(b.shape)
print(b.head())
print(b.columns.tolist())

(1115113, 16)
  match_id        date                                      venue match_type  \
0  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal        T20   
1  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal        T20   
2  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal        T20   
3  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal        T20   
4  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal        T20   

  gender               winner result method  innings  over  \
0   male  Sunrisers Hyderabad    NaN    NaN        1     0   
1   male  Sunrisers Hyderabad    NaN    NaN        1     0   
2   male  Sunrisers Hyderabad    NaN    NaN        1     0   
3   male  Sunrisers Hyderabad    NaN    NaN        1     0   
4   male  Sunrisers Hyderabad    NaN    NaN        1     0   

          batting_team                 bowling_team  runs_batter  runs_total  \
0  Sunrisers Hyderabad  Royal Challengers Bangalore 

In [6]:
b["batting_team"].value_counts()

batting_team
Mumbai Indians             34962
Pakistan                   33660
India                      32906
Kolkata Knight Riders      32439
Chennai Super Kings        32096
                           ...  
Turks and Caicos Island      422
Uzbekistan                   387
Chile                        261
Peru                         258
Iran                         131
Name: count, Length: 128, dtype: int64

In [7]:
TEAM_MAP = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

for col in ["batting_team", "bowling_team", "winner"]:
    b[col] = b[col].replace(TEAM_MAP)

In [8]:
b["venue"] = b["venue"].str.split(",").str[0].str.strip()

b["venue"].value_counts().head(20)

venue
Wankhede Stadium                                           35686
Dubai International Cricket Stadium                        33263
Eden Gardens                                               29165
MA Chidambaram Stadium                                     25744
Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)    25109
Rajiv Gandhi International Stadium                         22410
M Chinnaswamy Stadium                                      22136
Bayuemas Oval                                              18647
Udayana Cricket Ground                                     16850
Sawai Mansingh Stadium                                     16422
Gahanga International Cricket Stadium                      16170
Harare Sports Club                                         16072
Sheikh Zayed Stadium                                       15567
Marsa Sports Club                                          15068
Moara Vlasiei Cricket Ground                               14827
Feroz Shah Kotla   

In [9]:
tot = (
    b.groupby(["match_id", "innings"])["runs_total"]
    .sum()
    .unstack()
)

tot.head()

innings,1,2
match_id,,
1001349,168.0,172.0
1001351,173.0,176.0
1001353,187.0,146.0
1004729,169.0,129.0
1007655,170.0,168.0


In [10]:
m = (
    b.drop_duplicates("match_id")
    [["match_id", "date", "venue", "winner", "result", "method"]]
    .set_index("match_id")
)

In [11]:
f1 = (
    b[b["innings"] == 1]
    .drop_duplicates("match_id")
    .set_index("match_id")
)

m["team_bat_first"] = f1["batting_team"]
m["team_chase"] = f1["bowling_team"]

m["first_total"] = tot[1]
m["second_total"] = tot[2]

m["chase_won"] = (m["winner"] == m["team_chase"]).astype(int)

m.head()

,date,venue,winner,result,method,team_bat_first,team_chase,first_total,second_total,chase_won
match_id,,,,,,,,,,
1082591,2017-04-05,Rajiv Gandhi International Stadium,Sunrisers Hyderabad,NaN,NaN,Sunrisers Hyderabad,Royal Challengers Bengaluru,207.0,172.0,0
1082592,2017-04-06,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,NaN,NaN,Mumbai Indians,Rising Pune Supergiant,184.0,187.0,1
1082593,2017-04-07,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,NaN,NaN,Gujarat Lions,Kolkata Knight Riders,183.0,184.0,1
1082594,2017-04-08,Holkar Cricket Stadium,Punjab Kings,NaN,NaN,Rising Pune Supergiant,Punjab Kings,163.0,164.0,1
1082595,2017-04-08,M.Chinnaswamy Stadium,Royal Challengers Bengaluru,NaN,NaN,Royal Challengers Bengaluru,Delhi Capitals,157.0,142.0,0


In [12]:
has_winner = m["winner"].notna()
not_tie_or_no_result = m["result"].isna()
not_dl = m["method"].isna()
has_both_innings = m["first_total"].notna() & m["second_total"].notna()

# Count exclusions sequentially so each dropped match is counted once.
remaining = pd.Series(True, index=m.index)

drop_no_winner = remaining & ~has_winner
remaining &= has_winner

drop_tie_or_no_result = remaining & ~not_tie_or_no_result
remaining &= not_tie_or_no_result

drop_dl = remaining & ~not_dl
remaining &= not_dl

drop_missing_innings = remaining & ~has_both_innings
remaining &= has_both_innings

print("Dropped — no winner:", int(drop_no_winner.sum()))
print("Dropped — tie/no-result:", int(drop_tie_or_no_result.sum()))
print("Dropped — D/L method:", int(drop_dl.sum()))
print("Dropped — missing innings:", int(drop_missing_innings.sum()))
print("Kept:", int(remaining.sum()))

m = m[remaining].reset_index()

Dropped — no winner: 144
Dropped — tie/no-result: 0
Dropped — D/L method: 150
Dropped — missing innings: 0
Kept: 4595


In [13]:
from pathlib import Path

output_path = Path("data/interim/matches.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

m.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Shape:", m.shape)
print(m.head())

Saved: data\interim\matches.parquet
Shape: (4595, 11)
  match_id        date                                    venue  \
0  1082591  2017-04-05       Rajiv Gandhi International Stadium   
1  1082592  2017-04-06  Maharashtra Cricket Association Stadium   
2  1082593  2017-04-07   Saurashtra Cricket Association Stadium   
3  1082594  2017-04-08                   Holkar Cricket Stadium   
4  1082595  2017-04-08                    M.Chinnaswamy Stadium   

                        winner result method               team_bat_first  \
0          Sunrisers Hyderabad    NaN    NaN          Sunrisers Hyderabad   
1       Rising Pune Supergiant    NaN    NaN               Mumbai Indians   
2        Kolkata Knight Riders    NaN    NaN                Gujarat Lions   
3                 Punjab Kings    NaN    NaN       Rising Pune Supergiant   
4  Royal Challengers Bengaluru    NaN    NaN  Royal Challengers Bengaluru   

                    team_chase  first_total  second_total  chase_won  
0  Royal 

In [14]:
print("Matches:", len(m))
print("Duplicate match IDs:", m["match_id"].duplicated().sum())
print("Missing winners:", m["winner"].isna().sum())
print("Missing first totals:", m["first_total"].isna().sum())
print("Missing second totals:", m["second_total"].isna().sum())
print(m["chase_won"].value_counts())

Matches: 4595
Duplicate match IDs: 0
Missing winners: 0
Missing first totals: 0
Missing second totals: 0
chase_won
1    2379
0    2216
Name: count, dtype: int64


In [15]:
g = b.groupby(["match_id", "innings"], sort=False)

b["runs"] = g["runs_total"].cumsum()
b["wkts"] = g["is_wicket"].cumsum()
b["balls_bowled"] = g["is_legal"].cumsum()

# Keep legal deliveries only
s = b[b["is_legal"] == 1].copy()

s["balls_left"] = 120 - s["balls_bowled"]
s["wickets_in_hand"] = 10 - s["wkts"]

print(s[[
    "match_id", "innings", "runs", "wkts",
    "balls_bowled", "balls_left", "wickets_in_hand"
]].head())


  match_id  innings  runs  wkts  balls_bowled  balls_left  wickets_in_hand
0  1082591        1     0     0             1         119               10
1  1082591        1     0     0             2         118               10
2  1082591        1     4     0             3         117               10
3  1082591        1     4     0             4         116               10
5  1082591        1     6     0             5         115               10


In [16]:
keep = [
    "match_id", "team_bat_first", "team_chase",
    "first_total", "chase_won"
]

s = s.merge(
    m[keep],
    on="match_id",
    how="inner",
    validate="many_to_one"
)

# Innings 1: predict final score and whether the team batting first won
first = s[s["innings"] == 1].copy()
first["final_total"] = first["first_total"]
first["label_bat_first_won"] = 1 - first["chase_won"]

# Innings 2: predict whether the chasing team wins
chase = s[s["innings"] == 2].copy()
chase["target"] = chase["first_total"] + 1
chase["label"] = chase["chase_won"]

print("First-innings rows:", len(first))
print("Chase rows:", len(chase))
print("Chase win-label average:", chase["label"].mean())


First-innings rows: 539806
Chase rows: 482401
Chase win-label average: 0.4789832525222792


In [17]:

print("Negative balls_left — first:", (first["balls_left"] < 0).sum())
print("Negative balls_left — chase:", (chase["balls_left"] < 0).sum())

print("First-innings win-label average:",
      first["label_bat_first_won"].mean())

print("Chase win-label average:",
      chase["label"].mean())

print("Missing chase targets:", chase["target"].isna().sum())

# Inspect the final state of one chase
match_id = chase["match_id"].iloc[0]

print(
    chase[chase["match_id"] == match_id][[
        "match_id", "runs", "wkts", "balls_bowled",
        "balls_left", "target", "label"
    ]].tail()
)


Negative balls_left — first: 13
Negative balls_left — chase: 4
First-innings win-label average: 0.48741214436297486
Chase win-label average: 0.4789832525222792
Missing chase targets: 0
    match_id  runs  wkts  balls_bowled  balls_left  target  label
233  1082591   165     9           114           6   208.0      0
234  1082591   165     9           115           5   208.0      0
235  1082591   166     9           116           4   208.0      0
236  1082591   172     9           117           3   208.0      0
237  1082591   172    10           118           2   208.0      0


In [19]:
bad = s[s["balls_left"] < 0]

print(
    bad[[
        "match_id", "innings", "date", "venue",
        "balls_bowled", "balls_left"
    ]]
    .drop_duplicates(["match_id", "innings"])
    .to_string(index=False)
)

match_id  innings       date                                                   venue  balls_bowled  balls_left
  419155        1 2010-04-15                                  MA Chidambaram Stadium           121          -1
 1279383        1 2021-10-07                                     Happy Valley Ground           121          -1
 1286675        1 2021-11-10                             Sir Vivian Richards Stadium           121          -1
 1321263        1 2022-07-13                                Tikkurila Cricket Ground           121          -1
 1321465        2 2022-07-11                                      Queens Sports Club           121          -1
 1344794        2 2022-11-21 Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)           121          -1
 1384593        2 2023-07-14                                       Marsa Sports Club           121          -1
 1444955        1 2024-08-19                                       Faleata Oval No 2           121          -1
 

In [23]:
from pathlib import Path

out = Path("data/interim")
out.mkdir(parents=True, exist_ok=True)

first.to_parquet(out / "first_states.parquet", index=False)
chase.to_parquet(out / "chase_states.parquet", index=False)

print("Saved both state datasets.")

Saved both state datasets.
